# Lab 7.9 &mdash; Human-in-the-Loop: Draft ≠ Send

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Build a REAL gather-only create_agent -- send_email withheld
- Invoke it to draft, then gate the send: approve -> send, reject -> revise
- See that the strongest guardrail is structural: the tool isn't bound

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Human-in-the-loop: draft ≠ send](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-07-09")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The golden rule for real-world agents: **separate drafting from sending** (deck slides 13, 16). The
agent gathers, reasons and drafts **autonomously** &mdash; none of that is irreversible &mdash; but the
**send** pauses for a human. The simplest, strongest guardrail is to **withhold the `send_email`
tool**: you build the **real `create_agent`** with gather tools only, so the agent literally **cannot
send**. A human does that after approving. **Draft is not send.** (Lab 11 assembles this same
gather-only guardrail into the full email agent.)

In [ ]:
# The agent's candidate toolset -- notice what is deliberately WITHHELD.
CANDIDATE_TOOLS = ["lookup_order", "get_template", "send_email"]  # send_email must NOT be bound
print("what the agent COULD be given:", CANDIDATE_TOOLS)

## Build it
Complete the **gather-only agent** (bind `[lookup_order]`, never `send_email`), the draft flag, and the
approval gate.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order's status, ETA and carrier by id."""
    return ORDERS.get(order_id, {"status": "unknown"})

@tool
def send_email(to: str, body: str) -> str:
    """Send an email. (Defined to show the capability -- but DELIBERATELY WITHHELD from the agent.)"""
    return "SENT"

def gather_only_agent():
    # the strongest guardrail: build the REAL agent WITHOUT send_email
    return create_agent(llm, ___)   # TODO: gather-only tools -- [lookup_order], NOT send_email

def make_draft(reply):
    # the agent's output is a DRAFT + a needs-approval flag -- never a sent mail
    return {"reply": reply, "status": ___}   # TODO: the needs-approval flag

def gate(draft, approved):
    # the human-in-the-loop gate: approve -> send, reject -> revise
    if draft["status"] != "needs_approval":
        return "invalid"
    return ___   # TODO: send when approved, otherwise revise

try:
    # Rule-based draft/gate -- no model call yet:
    d = make_draft("Hi Priya, your order 4471 is due Friday.")
    print("draft   :", d)
    print("approve ->", gate(d, True))
    print("reject  ->", gate(d, False))
    print("send_email is DEFINED but will NOT be bound to the agent:", send_email.name)
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Build the real gather-only agent, invoke it to draft a reply, then run the human gate. Read `tools_used`: it calls `lookup_order` and never has a send tool to call.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to the repo .env (free key at console.groq.com), then re-run this cell.")
else:
    try:
        def tools_used(messages):
            return [tc["name"] for m in messages for tc in (getattr(m, "tool_calls", None) or [])]

        agent = gather_only_agent()          # a REAL create_agent, send_email NOT bound
        result = with_backoff(lambda: agent.invoke(
            {"messages": [("user", "Look up order 4471 and draft a one-line status reply. Do not send anything.")]},
            config={"recursion_limit": 8}))
        reply = result["messages"][-1].content
        print("tools the agent used:", tools_used(result["messages"]))   # lookup_order -- never send_email

        draft = make_draft(reply)            # wrap the real draft as needs_approval
        print("draft status  :", draft["status"])
        print("human approves ->", gate(draft, True))   # only NOW, past a human, does 'send' happen
        print("could the agent auto-send? No -- send_email was never bound.")
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `gather_only_agent()` is a **real `create_agent`** built with **only** `lookup_order` &mdash; `send_email` is defined but **never bound**, so the agent structurally **cannot send**.
- The real draft is wrapped **`needs_approval`**; `gate` routes approve&rarr;send / reject&rarr;revise &mdash; the *send* only ever happens past a human.
- `tools_used` from the trace shows `lookup_order` and never `send_email` &mdash; the guardrail is what's **absent** from the tools list.

## Your turn (open task &mdash; no grader)
Add a third state to `gate` &mdash; e.g. `"edit"` when a human tweaks the reply before sending &mdash; and
decide what `status` an edited draft carries. **What good looks like:** every path still ends with a human
action for the send, and there is no code path where the agent sends on its own.

---
### Key takeaway
The strongest human-in-the-loop guardrail is the simplest: build the real agent without the send tool. It gathers and drafts all day -- and a human keeps the send. Draft is not send.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>